In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import firebase_admin
from firebase_admin import credentials, db


# ==========================================
# 1. FIREBASE INITIALIZATION
# ==========================================
print("--- Initializing Firebase ---")
JSON_FILE = "hydrosmart-pbl-firebase-adminsdk-fbsvc-edd5c9e5f9.json"

DATABASE_URL = "https://hydrosmart-pbl-default-rtdb.asia-southeast1.firebasedatabase.app/"

# Clear any existing app instances to apply the new URL safely
if firebase_admin._apps:
    for app_name in list(firebase_admin._apps):
        firebase_admin.delete_app(firebase_admin._apps[app_name])

try:
    cred = credentials.Certificate(JSON_FILE)
    firebase_admin.initialize_app(cred, {'databaseURL': DATABASE_URL})
    print("✅ Firebase initialized successfully.\n")
except Exception as e:
    print(f"❌ Firebase Initialization Error: {e}")
    exit()

# ==========================================
# 2. LOAD THE TRAINED AI & SCALERS
# ==========================================
print("--- Loading AI Model and Scalers ---")
try:
    model = tf.keras.models.load_model('hydroponics_ai_model.keras')
    scaler_X = joblib.load('scaler_features.pkl')
    scaler_y = joblib.load('scaler_targets.pkl')
    print("✅ Successfully loaded model and scalers!\n")
except Exception as e:
    print(f"❌ ERROR: Could not load files. ({e})")
    exit()

# ==========================================
# 3. COLLECT 15-STEP LIVE INPUT FROM FIREBASE
# ==========================================
print("--- Collecting Live Data Buffer from Firebase ---")
features = ['water_level', 'DHT_temp', 'TDS', 'pH', 'DHT_humidity']
live_buffer = {feature: [] for feature in features}

# The LSTM needs 15 sequential readings to analyze momentum.
required_steps = 15
print(f"Buffering {required_steps} sequential readings from live sensors...")

for step in range(required_steps):
    # Retrieve live sensor data from the 'sensors' node
    sensors_ref = db.reference('sensors')
    data = sensors_ref.get()

    if data:
        # Safely extract values based on the database structure
        w_level = data.get('water_level', 0)
        a_temp = data.get('ambient', {}).get('temp', 0)
        tds_val = data.get('water', {}).get('tds', 0)
        # Defaulting pH to 7.0 in case it is temporarily missing from the stream
        ph_val = data.get('water', {}).get('ph', 7.0)
        a_hum = data.get('ambient', {}).get('humidity', 0)

        live_buffer['water_level'].append(w_level)
        live_buffer['DHT_temp'].append(a_temp)
        live_buffer['TDS'].append(tds_val)
        live_buffer['pH'].append(ph_val)
        live_buffer['DHT_humidity'].append(a_hum)

        print(f"  ↳ Step {step+1}/{required_steps} captured.")
    else:
        print("  ↳ Warning: Failed to retrieve sensor data.")

    # Delay to build a realistic time-series window (adjustable based on ESP32 upload rate)
    time.sleep(2)

# ==========================================
# 4. SHAPE AND SCALE THE DATA FOR LSTM
# ==========================================
input_df = pd.DataFrame(live_buffer)
scaled_input = scaler_X.transform(input_df)
# Clip prevents sudden database noise from spiking beyond the 0-1 trained boundaries
scaled_input = np.clip(scaled_input, 0.0, 1.0)
lstm_input = np.reshape(scaled_input, (1, 15, 5))

# ==========================================
# 5. AI INFERENCE (PREDICT THE FUTURE)
# ==========================================
print("\nAnalyzing chemical momentum...")
scaled_prediction = model.predict(lstm_input, verbose=0)
real_prediction = scaler_y.inverse_transform(scaled_prediction)

pred_water = real_prediction[0][0]
pred_temp  = real_prediction[0][1]
pred_tds   = real_prediction[0][2]
pred_ph    = real_prediction[0][3]
pred_hum   = real_prediction[0][4]

print("\n=============================================")
print("     PREDICTED SYSTEM STATE (IN 15 MIN)      ")
print("=============================================")
print(f"Water Level : {pred_water:.2f}")
print(f"Temperature : {pred_temp:.2f} °C")
print(f"TDS         : {pred_tds:.2f} PPM")
print(f"pH Level    : {pred_ph:.2f}")
print(f"Humidity    : {pred_hum:.2f} %")
print("=============================================\n")

# ==========================================
# 6. EDGE ACTUATION LOGIC (DECISION MAKING)
# ==========================================
print("--- AUTOMATED PUMP DECISIONS ---")

# Initialize pump states for Firebase (0 = OFF, 1 = ON)
pump_states = {
    'nutrient_pump': 0,
    'freshwater_valve': 0,
    'ph_down_pump': 0,
    'ph_up_pump': 0
}

# Nutrient Dosing Logic
if pred_tds < 800:
    print("⚠️  ALERT: TDS dropping below minimum threshold.")
    print("➤  ACTION: OPEN NUTRIENT PUMP (Preemptive Micro-dose)")
    pump_states['nutrient_pump'] = 1
elif pred_tds > 1200:
    print("⚠️  ALERT: TDS exceeding maximum threshold (Toxicity Risk).")
    print("➤  ACTION: OPEN FRESHWATER VALVE (Dilution)")
    pump_states['freshwater_valve'] = 1
else:
    print("✅  TDS is stable. No pump action required.")

# pH Balancing Logic
if pred_ph > 6.5:
    print("⚠️  ALERT: pH rising above 6.5 (Alkaline drift).")
    print("➤  ACTION: OPEN 'pH DOWN' PUMP")
    pump_states['ph_down_pump'] = 1
elif pred_ph < 5.5:
    print("⚠️  ALERT: pH falling below 5.5 (Acidic drift).")
    print("➤  ACTION: OPEN 'pH UP' PUMP")
    pump_states['ph_up_pump'] = 1
else:
    print("✅  pH is stable. No pump action required.")

# ==========================================
# 7. FIREBASE OVERRIDE & UPDATE
# ==========================================
print("\n--- Sending Commands to Firebase ---")
try:
    # Target the system_control node to push predictions
    db_ref_predictions = db.reference('system_control/predictions')
    db_ref_predictions.set({
        'predicted_water': float(pred_water),
        'predicted_temp': float(pred_temp),
        'predicted_tds': float(pred_tds),
        'predicted_ph': float(pred_ph),
        'predicted_humidity': float(pred_hum)
    })

    # Target the pumps node to trigger the ESP32 hardware
    db_ref_pumps = db.reference('pumps')
    db_ref_pumps.update(pump_states)

    # Update timestamp
    db_ref_ts = db.reference('system_control')
    db_ref_ts.update({'last_update_ts': {'.sv': 'timestamp'}})

    print("✅ SUCCESS: Database and Pumps Updated!")
except Exception as e:
    print(f"❌ Error updating Firebase: {e}")

--- Initializing Firebase ---
✅ Firebase initialized successfully.

--- Loading AI Model and Scalers ---
❌ ERROR: Could not load files. (File not found: filepath=hydroponics_ai_model.keras. Please ensure the file is an accessible `.keras` zip file.)
--- Collecting Live Data Buffer from Firebase ---
Buffering 15 sequential readings from live sensors...
  ↳ Step 1/15 captured.
  ↳ Step 2/15 captured.
  ↳ Step 3/15 captured.
  ↳ Step 4/15 captured.
  ↳ Step 5/15 captured.
  ↳ Step 6/15 captured.
  ↳ Step 7/15 captured.
  ↳ Step 8/15 captured.
  ↳ Step 9/15 captured.
  ↳ Step 10/15 captured.
  ↳ Step 11/15 captured.
  ↳ Step 12/15 captured.
  ↳ Step 13/15 captured.
  ↳ Step 14/15 captured.


KeyboardInterrupt: 